## UrbanRide_11 — Churn Prediction Model
**Type:** Binary Classification  
**Label:** `is_churned` (true = churned, false = active)  
**Source:** `urbanride.gold.customer_features`  
**Schedule:** Weekly · Sunday 04:00 AM (Job 4)

**Models trained:**
- Logistic Regression — baseline
- Random Forest — main model

**Class imbalance:** 12.1% churned vs 87.9% active — handled via `weightCol`  
**Split strategy:** Time-based on `signup_date` — prevents data leakage  
**Experiment:** `/urbanride_churn_prediction`  
**Registry:** `urbanride_churn_model`

**Key features:**
- `days_since_last_trip` — strongest churn signal (9 days active vs 72 days churned)
- `total_trips`, `total_spend`, `customer_age_days` — strong signals
- `city`, `device_type`, `referral_source`, `favourite_vehicle_type` — encoded categoricals
- `preferred_payment_method` dropped — zero variance (all UPI)

**Output:** Predictions written to `urbanride.gold.churn_predictions`  
**Feeds:** Customer Segmentation Model (14)

In [0]:
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, lit, current_timestamp
import pandas as pd

# Do NOT call mlflow.autolog() — crashes on serverless

CATALOG         = 'urbanride'
GOLD            = f'{CATALOG}.gold'
EXPERIMENT_NAME = '/urbanride_churn_prediction'
MODEL_NAME      = 'urbanride_churn_model'
TMP_DIR         = '/Volumes/urbanride/silver/mlflow_artifacts/mlflow_tmp'

# Create TMP_DIR if it doesn't exist — MLflow needs this to stage model files
dbutils.fs.mkdirs(TMP_DIR)
print(f'TMP_DIR ready: {TMP_DIR}')

# Feature columns
NUMERIC_COLS = [
    'days_since_last_trip',
    'total_trips',
    'completed_trips',
    'cancelled_trips',
    'cancellation_rate',
    'total_spend',
    'avg_spend_per_trip',
    'customer_age_days',
    'avg_trip_distance_km',
    'avg_trip_duration_minutes',
    'avg_surge_multiplier',
    'weekend_trip_ratio',
    'rainy_day_trip_ratio',
    'failed_payment_count',
    'failed_payment_rate',
]

CATEGORICAL_COLS = [
    'city',
    'device_type',
    'referral_source',
    'favourite_vehicle_type',
]

LABEL_COL  = 'label'
WEIGHT_COL = 'class_weight'

# Set MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'Catalog     : {CATALOG}')
print(f'Source      : {GOLD}.customer_features')
print(f'Experiment  : {EXPERIMENT_NAME}')
print(f'Model name  : {MODEL_NAME}')
print(f'Numeric features    : {len(NUMERIC_COLS)}')
print(f'Categorical features: {len(CATEGORICAL_COLS)}')
print('Config ready.')


In [0]:
print('Loading gold.customer_features...')

df_raw = spark.table(f'{GOLD}.customer_features')

print(f'  Total rows : {df_raw.count():,}')
print()

# Churn distribution
print('Churn distribution:')
df_raw.groupBy('is_churned').count().orderBy('is_churned').show()

# Select only needed columns
ALL_FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

df = df_raw.select(
    ['customer_id', 'signup_date', 'is_churned'] + ALL_FEATURE_COLS
).withColumn(
    # Convert boolean label to integer for Spark ML
    LABEL_COL,
    col('is_churned').cast('int')
)

# Drop rows with null label
df = df.filter(col(LABEL_COL).isNotNull())

print(f'  Rows after null label filter : {df.count():,}')


In [0]:
# Time-based split on signup_date
# Train: customers who signed up before Nov 1 2025
# Test:  customers who signed up on or after Nov 1 2025
# Reason: prevents data leakage — model never sees future customers during training

from pyspark.sql.functions import to_date

SPLIT_DATE = '2026-01-01'

df_train = df.filter(col('signup_date') <  SPLIT_DATE)
df_test  = df.filter(col('signup_date') >= SPLIT_DATE)

train_count = df_train.count()
test_count  = df_test.count()
total       = train_count + test_count

print(f'Split date : {SPLIT_DATE}')
print(f'Train rows : {train_count:,} ({train_count/total*100:.1f}%)')
print(f'Test rows  : {test_count:,}  ({test_count/total*100:.1f}%)')
print()

# Verify churn rate is similar in both splits — important sanity check
print('Churn rate in train set:')
df_train.groupBy('is_churned').count().orderBy('is_churned').show()
print('Churn rate in test set:')
df_test.groupBy('is_churned').count().orderBy('is_churned').show()


In [0]:
# Class imbalance: 12.1% churned vs 87.9% active
# Without class weights, model will predict 'active' for everything and get 87.9% accuracy
# Class weight formula: weight = total / (num_classes * class_count)
# This makes the minority class (churned) count more during training

print('Computing class weights...')

train_total   = df_train.count()
churned_count = df_train.filter(col(LABEL_COL) == 1).count()
active_count  = df_train.filter(col(LABEL_COL) == 0).count()
num_classes   = 2

weight_churned = train_total / (num_classes * churned_count)
weight_active  = train_total / (num_classes * active_count)

print(f'  Train total    : {train_total:,}')
print(f'  Churned count  : {churned_count:,}')
print(f'  Active count   : {active_count:,}')
print(f'  Weight churned : {weight_churned:.4f}')
print(f'  Weight active  : {weight_active:.4f}')

# Add class weight column to train set
df_train = df_train.withColumn(
    WEIGHT_COL,
    when(col(LABEL_COL) == 1, weight_churned)
    .otherwise(weight_active)
)

# Test set does not need weights — weights only used during training
df_test = df_test.withColumn(WEIGHT_COL, lit(1.0))

print('Class weights added to train set.')


In [0]:
print('Building feature pipeline...')

# Step 1: StringIndexer for each categorical column
# Converts string categories to numeric indices
# e.g. city: Delhi=0, Mumbai=1, Bengaluru=2 etc.
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f'{c}_idx',
        handleInvalid='keep'   # unseen categories get a new index
    )
    for c in CATEGORICAL_COLS
]

# Step 2: VectorAssembler — combine all features into one vector
indexed_cat_cols = [f'{c}_idx' for c in CATEGORICAL_COLS]
ALL_INPUT_COLS   = NUMERIC_COLS + indexed_cat_cols

assembler = VectorAssembler(
    inputCols=ALL_INPUT_COLS,
    outputCol='features_raw',
    handleInvalid='skip'
)

# Step 3: StandardScaler — needed for Logistic Regression
# Features have very different ranges (days_since_last_trip vs total_spend)
# Scaler brings them to same range so LR doesn't get confused by magnitude
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True
)

print(f'  Numeric features    : {len(NUMERIC_COLS)}')
print(f'  Categorical features: {len(CATEGORICAL_COLS)} → {len(indexed_cat_cols)} indexed')
print(f'  Total features      : {len(ALL_INPUT_COLS)}')
print('Feature pipeline ready.')


In [0]:
print('Training Logistic Regression (baseline)...')

lr = LogisticRegression(
    featuresCol  = 'features',
    labelCol     = LABEL_COL,
    weightCol    = WEIGHT_COL,
    maxIter      = 20,
    regParam     = 0.01,
)

# Pipeline: indexers → assembler → scaler → LR
lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

auc_evaluator = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, metricName='areaUnderROC'
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL, metricName='f1'
)

with mlflow.start_run(run_name='logistic_regression') as run:
    lr_run_id = run.info.run_id

    # Train
    lr_model = lr_pipeline.fit(df_train)
    lr_preds = lr_model.transform(df_test)

    # Evaluate
    lr_auc = auc_evaluator.evaluate(lr_preds)
    lr_f1  = f1_evaluator.evaluate(lr_preds)

    # Log to MLflow
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('max_iter',   20)
    mlflow.log_param('reg_param',  0.01)
    mlflow.log_param('split_date', '2026-01-01')
    mlflow.log_param('train_rows', df_train.count())
    mlflow.log_param('test_rows',  df_test.count())
    mlflow.log_metric('auc', lr_auc)
    mlflow.log_metric('f1',  lr_f1)

print(f'  LR AUC : {lr_auc:.4f}')
print(f'  LR F1  : {lr_f1:.4f}')


In [0]:
print('Training Random Forest...')

# RF does NOT need StandardScaler — trees are scale-invariant
# Use features_raw directly from assembler
assembler_rf = VectorAssembler(
    inputCols=ALL_INPUT_COLS,
    outputCol='features',
    handleInvalid='skip'
)

param_combos = [
    {'numTrees': 50,  'maxDepth': 5},
    {'numTrees': 50,  'maxDepth': 10},
    {'numTrees': 100, 'maxDepth': 5},
    {'numTrees': 100, 'maxDepth': 10},
]

best_rf_auc    = 0
best_rf_params = None
best_rf_run_id = None
best_rf_model  = None

for params in param_combos:
    run_name = f'rf_trees{params["numTrees"]}_depth{params["maxDepth"]}'

    rf = RandomForestClassifier(
        featuresCol = 'features',
        labelCol    = LABEL_COL,
        weightCol   = WEIGHT_COL,
        numTrees    = params['numTrees'],
        maxDepth    = params['maxDepth'],
        seed        = 42,
    )

    rf_pipeline = Pipeline(stages=indexers + [assembler_rf, rf])

    with mlflow.start_run(run_name=run_name) as run:
        rf_model = rf_pipeline.fit(df_train)
        rf_preds = rf_model.transform(df_test)

        rf_auc = auc_evaluator.evaluate(rf_preds)
        rf_f1  = f1_evaluator.evaluate(rf_preds)

        mlflow.log_param('model_type', 'RandomForest')
        mlflow.log_param('num_trees',  params['numTrees'])
        mlflow.log_param('max_depth',  params['maxDepth'])
        mlflow.log_param('split_date', '2026-01-01')
        mlflow.log_metric('auc', rf_auc)
        mlflow.log_metric('f1',  rf_f1)

        print(f'  {run_name:<35} AUC={rf_auc:.4f}  F1={rf_f1:.4f}')

        if rf_auc > best_rf_auc:
            best_rf_auc    = rf_auc
            best_rf_params = params
            best_rf_run_id = run.info.run_id
            best_rf_model  = rf_model

print()
print(f'  Best RF params : {best_rf_params}')
print(f'  Best RF AUC    : {best_rf_auc:.4f}')


In [0]:
print('='*55)
print('CHURN MODEL COMPARISON')
print('='*55)
print(f'{"Model":<40} {"AUC":>8} {"F1":>8}')
print('-'*55)
print(f'{"Logistic Regression":<40} {lr_auc:>8.4f} {lr_f1:>8.4f}')
print(f'{"Random Forest (best: " + str(best_rf_params) + ")":<40} {best_rf_auc:>8.4f}')
print('='*55)

# Pick best model — RF wins if AUC > LR
if best_rf_auc >= lr_auc:
    BEST_MODEL    = best_rf_model
    BEST_RUN_ID   = best_rf_run_id
    BEST_AUC      = best_rf_auc
    BEST_MODEL_TYPE = f'RandomForest_{best_rf_params}'
    print('\nWinner: Random Forest')
else:
    BEST_MODEL    = lr_model
    BEST_RUN_ID   = lr_run_id
    BEST_AUC      = lr_auc
    BEST_MODEL_TYPE = 'LogisticRegression'
    print('\nWinner: Logistic Regression')

print(f'Best AUC    : {BEST_AUC:.4f}')
print(f'Best run ID : {BEST_RUN_ID}')
print(f'\nCheck Experiments UI: {EXPERIMENT_NAME}')


In [0]:
print('Registering best model to MLflow Model Registry...')

client = MlflowClient()

# Create model in registry if not exists
try:
    client.create_registered_model(
        name        = MODEL_NAME,
        description = 'Predicts customer churn for ZipCab ride-hailing platform.'
    )
    print(f'  Created model: {MODEL_NAME}')
except Exception:
    print(f'  Model already exists: {MODEL_NAME} — adding new version')

# Build signature — required by Unity Catalog
sample_input  = df_test.select(ALL_FEATURE_COLS).limit(100)
sample_pandas = sample_input.toPandas()

sample_preds  = BEST_MODEL.transform(sample_input)
sample_output = (
    sample_preds
    .withColumn('churn_probability', vector_to_array('probability')[1])
    .select('prediction', 'churn_probability')
    .toPandas()
)

signature = infer_signature(
    model_input  = sample_pandas,
    model_output = sample_output
)
print('  Signature inferred.')

# Log and register best model
with mlflow.start_run(run_name='register_churn_model') as reg_run:
    mlflow.log_param('model_type',   BEST_MODEL_TYPE)
    mlflow.log_param('best_auc',     BEST_AUC)
    mlflow.log_param('train_rows',   df_train.count())
    mlflow.log_param('feature_count',len(ALL_INPUT_COLS))
    mlflow.log_metric('auc', BEST_AUC)

    # Log feature importance as artifact (RF only)
    if 'RandomForest' in BEST_MODEL_TYPE:
        rf_stage = BEST_MODEL.stages[-1]
        fi_df = pd.DataFrame({
            'feature':    ALL_INPUT_COLS,
            'importance': rf_stage.featureImportances.toArray()
        }).sort_values('importance', ascending=False)
        import tempfile, os
        with tempfile.TemporaryDirectory() as tmp:
            fi_path = os.path.join(tmp, 'churn_feature_importance.csv')
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)
        print('  Feature importance logged.')

    mlflow.spark.log_model(
        spark_model   = BEST_MODEL,
        artifact_path = 'churn_model',
        signature     = signature,
        input_example = sample_pandas.head(5),
        dfs_tmpdir    = TMP_DIR
    )

    model_uri = f'runs:/{reg_run.info.run_id}/churn_model'
    mv = mlflow.register_model(model_uri, MODEL_NAME)

# Add version description
client.update_model_version(
    name        = MODEL_NAME,
    version     = mv.version,
    description = (
        f'{BEST_MODEL_TYPE}. AUC={BEST_AUC:.4f}. '
        f'Trained on ZipCab 6 months data. 89K customers. '
        f'Key feature: days_since_last_trip.'
    )
)

# Transition to Production
client.transition_model_version_stage(
    name    = MODEL_NAME,
    version = mv.version,
    stage   = 'Production'
)

print(f'  Model registered   : {MODEL_NAME}')
print(f'  Version            : {mv.version}')
print(f'  Stage              : Production')
print(f'  Check Models UI    : {MODEL_NAME}')


In [0]:
print('Saving churn predictions to Delta table...')

# Run model on full dataset — score all customers
df_all = df.withColumn(WEIGHT_COL, lit(1.0))
df_scored = BEST_MODEL.transform(df_all)

# Extract churn probability from probability vector
df_predictions = df_scored.select(
    col('customer_id'),
    col('city'),
    col('is_churned'),
    col(LABEL_COL).alias('actual_label'),
    col('prediction').alias('predicted_label'),
    vector_to_array('probability')[1].alias('churn_probability'),

    # Risk tier — useful for business actions
    when(vector_to_array('probability')[1] >= 0.7, 'high_risk')
    .when(vector_to_array('probability')[1] >= 0.4, 'medium_risk')
    .otherwise('low_risk').alias('churn_risk_tier'),

).withColumn('_processed_at', current_timestamp()) \
 .withColumn('_model_version', lit(str(mv.version))) \
 .withColumn('_model_name', lit(MODEL_NAME))

# Write predictions
df_predictions.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD}.churn_predictions')

pred_count = df_predictions.count()
print(f'  Predictions written : {pred_count:,}')
print(f'  Table               : {GOLD}.churn_predictions')


In [0]:
print('=== CHURN MODEL VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.churn_predictions')
total = df_verify.count()

print(f'  Total predictions : {total:,}')
print(f'  Model             : {BEST_MODEL_TYPE}')
print(f'  Best AUC          : {BEST_AUC:.4f}')
print()

print('Risk tier distribution:')
df_verify.groupBy('churn_risk_tier').count().orderBy('count', ascending=False).show()

print('Prediction vs actual:')
df_verify.groupBy('actual_label', 'predicted_label').count().orderBy('actual_label', 'predicted_label').show()

print('High risk customers by city:')
df_verify.filter(col('churn_risk_tier') == 'high_risk') \
    .groupBy('city').count().orderBy('count', ascending=False).show()

print('Avg churn probability by actual churn:')
from pyspark.sql.functions import avg, round as spark_round
df_verify.groupBy('is_churned').agg(
    spark_round(avg('churn_probability'), 4).alias('avg_churn_probability')
).orderBy('is_churned').show()

# Feature importance (RF only)
if 'RandomForest' in BEST_MODEL_TYPE:
    print('Top 10 most important features:')
    rf_stage = BEST_MODEL.stages[-1]
    fi = sorted(
        zip(ALL_INPUT_COLS, rf_stage.featureImportances.toArray()),
        key=lambda x: x[1],
        reverse=True
    )
    print(f'  {"Feature":<35} {"Importance":>10}')
    print(f'  {"-"*47}')
    for feat, imp in fi[:10]:
        print(f'  {feat:<35} {imp:>10.4f}')

print()
print('Sample high risk customers:')
df_verify.filter(col('churn_risk_tier') == 'high_risk').select(
    'customer_id', 'city', 'is_churned',
    'churn_probability', 'churn_risk_tier'
).limit(5).show(truncate=False)

print()
print(f'Churn model complete.')
print(f'Check Experiments UI : {EXPERIMENT_NAME}')
print(f'Check Models UI      : {MODEL_NAME}')
print(f'Predictions table    : {GOLD}.churn_predictions')
print('Next: UrbanRide_12 — Fraud Detection Model')
